# Daum Movie Review Notebook

In [20]:
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split

from preprocessing import DEFAULT_STOPWORDS, KoreanSentimentDataset, preprocess_dataframe
from model import BiLSTMSentimentClassifier

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cpu')

In [21]:
url = 'https://raw.githubusercontent.com/skn29teacher/skn29_lecture/main/data_nlp/daum_movie_review.csv'
df = pd.read_csv(url)
df.head()

,review,rating,date,title
0,돈 들인건 티가 나지만 보는 내내 하품만,1,2018.10.29,인피니티 워
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,2018.10.26,인피니티 워
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,2018.10.24,인피니티 워
3,이 정도면 볼만하다고 할 수 있음!,8,2018.10.22,인피니티 워
4,재미있다,10,2018.10.20,인피니티 워


In [22]:
print(df.columns)

review_col = 'review'
rating_col = 'rating'

data = df[[review_col, rating_col]].copy()
data.columns = ['review', 'rating']
data['sentiment'] = (data['rating'] >= 6).astype(int)
data[['review', 'rating', 'sentiment']].head()

Index(['review', 'rating', 'date', 'title'], dtype='object')


,review,rating,sentiment
0,돈 들인건 티가 나지만 보는 내내 하품만,1,0
1,몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.,10,1
2,이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...,8,1
3,이 정도면 볼만하다고 할 수 있음!,8,1
4,재미있다,10,1


In [23]:
processed_df, vocab = preprocess_dataframe(
    df=data[['review', 'sentiment']],
    text_col='review',
    label_col='sentiment',
    max_len=30,
    stopwords=DEFAULT_STOPWORDS,
    min_freq=1,
)

dataset = KoreanSentimentDataset(processed_df)
print('vocab size:', len(vocab))
print('dataset size:', len(dataset))
print(processed_df[['review', 'clean_text', 'tokens', 'padded_ids', 'length', 'label']].head())

vocab size: 54967
dataset size: 14725
                                              review  \
0                             돈 들인건 티가 나지만 보는 내내 하품만   
1       몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만.... 전국 맛집의 음식들을 한데 ...   
3                                이 정도면 볼만하다고 할 수 있음!   
4                                               재미있다   

                                          clean_text  \
0                             돈 들인건 티가 나지만 보는 내내 하품만   
1          몰입할수밖에 없다 어렵게 생각할 필요없다 내가 전투에 참여한듯 손에 땀이남   
2  이전 작품에 비해 더 화려하고 스케일도 커졌지만 전국 맛집의 음식들을 한데 모은 것...   
3                                 이 정도면 볼만하다고 할 수 있음   
4                                               재미있다   

                                              tokens  \
0                     [돈, 들인건, 티가, 나지만, 보는, 내내, 하품만]   
1  [몰입할수밖에, 없다, 어렵게, 생각할, 필요없다, 내가, 전투에, 참여한듯, 손에...   
2  [이전, 작품에, 비해, 화려하고, 스케일도, 커졌지만, 전국, 맛집의, 음식들을,...   
3                                [정도면, 볼만하다고, 할, 있음]   
4       

In [34]:
# val_size = max(1, int(len(dataset) * 0.2)) if len(dataset) > 1 else 0
# train_size = len(dataset) - val_size

# if val_size > 0:
#     train_dataset, val_dataset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))
# else:
#     train_dataset = dataset
#     val_dataset = dataset

from sklearn.model_selection import train_test_split
train_dataset, other_dataset = train_test_split(dataset, random_state=42 ,test_size=0.2)
val_dataset, test_dataset = train_test_split(other_dataset,test_size=0.5,random_state=42)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)


print(f'train size : {len(train_loader)}, val size : {len(val_loader)}, test size : {len(test_loader)}')

model = BiLSTMSentimentClassifier(
    vocab_size=len(vocab),
    embedding_dim=64,
    hidden_dim=128,
    num_layers=1,
    dropout=0.3,
    pad_idx=vocab['<pad>'],
    verbose=False,
).to(device)

criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

model

train size : 369, val size : 46, test size : 47


BiLSTMSentimentClassifier(
  (embedding): Embedding(54967, 64, padding_idx=0)
  (bilstm): LSTM(64, 128, batch_first=True, bidirectional=True)
  (dropout): Dropout(p=0.3, inplace=False)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (sigmoid): Sigmoid()
)

In [35]:
from tqdm import tqdm
def accuracy_from_probs(probs: torch.Tensor, labels: torch.Tensor) -> float:
    preds = (probs >= 0.5).float()
    return (preds == labels).float().mean().item()

epochs = 2
tq = tqdm(range(epochs))
for epoch in tq:
    tq.set_description(f'Epoch - {epoch+1}')
    model.train()
    for batch_idx, (input_ids, labels, lengths) in enumerate(train_loader):
        input_ids = input_ids.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        lengths = lengths.to(device)

        optimizer.zero_grad(set_to_none=True)
        probs = model(input_ids, lengths)
        loss = criterion(probs, labels)
        loss.backward()
        optimizer.step()        
        
        # print('train loss:', float(loss.item()))
        # print('train acc :', accuracy_from_probs(probs.detach(), labels.detach()))    

    model.eval()
    with torch.no_grad():
        for batch_idx, (input_ids, labels, lengths) in enumerate(val_loader):
            input_ids = input_ids.to(device)
            labels = labels.to(device).float().unsqueeze(1)
            lengths = lengths.to(device)
            probs = model(input_ids, lengths)
            # print('val acc:', accuracy_from_probs(probs, labels))
            tq.set_postfix(train_loss = float(loss.item()), 
                           train_acc = accuracy_from_probs(probs.detach(), labels.detach()),
                           val_acc =  accuracy_from_probs(probs, labels)
                           )

Epoch - 2: 100%|██████████| 2/2 [01:52<00:00, 56.26s/it, train_acc=0.781, train_loss=0.159, val_acc=0.781]


In [ ]:
# 평가
model.eval()
with torch.no_grad():
    total_acc = 0
    for batch_idx, (input_ids, labels, lengths) in enumerate(test_loader):
        input_ids = input_ids.to(device)
        labels = labels.to(device).float().unsqueeze(1)
        lengths = lengths.to(device)
        probs = model(input_ids, lengths)
        total_acc += accuracy_from_probs(probs, labels)                
    print('test acc:', total_acc / len(test_loader))           

tensor([30,  2, 30,  4,  4,  4, 20, 30, 25,  2, 18, 16,  4,  4,  6, 17,  4, 30,
        24,  5,  4,  5,  3,  3, 30, 29, 13, 22, 30,  3,  2,  9])


In [57]:
processed_df['review'][:2].tolist()

['돈 들인건 티가 나지만 보는 내내 하품만', '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.']

In [89]:
from preprocessing import clean_text, keep_korean_only,tokenize,remove_stopwords, tokens_to_ids, DEFAULT_STOPWORDS,pad_sequence
test = ['돈 들인건 티가 나지만 보는 내내 하품만', '몰입할수밖에 없다. 어렵게 생각할 필요없다. 내가 전투에 참여한듯 손에 땀이남.']
# test_tensor = torch.tensor(processed_df["padded_ids"].tolist(), dtype=torch.long)
processing = [clean_text(t)  for t  in test]
processing = [ keep_korean_only(t) for t in processing]
processing = [ tokenize(t) for t in processing]
processing = [remove_stopwords(t, stopwords=DEFAULT_STOPWORDS) for t in processing]
test = [   
    pad_sequence(tokens_to_ids(processing[1], vocab), max_len=30, pad_idx=vocab["<pad>"])
    for t in processing]

test_tensor = torch.tensor(test, dtype=torch.long)

model.eval()
with torch.no_grad():
    lengths = torch.tensor(
        [ max(1, min(len(tokens), 30)) for tokens in processing],
        dtype=torch.long,
        device = test_tensor.device
    )
    probs = model(test_tensor, lengths)
    print(probs)

tensor([[0.9626],
        [0.9821]])
